In [2]:
pip install faker 

Note: you may need to restart the kernel to use updated packages.


In [6]:
import random
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)
fake = Faker()
Faker.seed(42)

def random_date(start, end):
    return start + timedelta(
        seconds=random.randint(0, int((end - start).total_seconds()))
    )

START = datetime(2021, 1, 1)
END   = datetime(2023, 12, 31)

customers = []
for i in range(1, 5001):
    customers.append({
        "customer_id"      : f"CUST{i:06d}",
        "first_name"       : fake.first_name(),
        "last_name"        : fake.last_name(),
        "age"              : int(np.clip(np.random.normal(42,14),18,80)),
        "email"            : fake.email(),
        "state"            : random.choice(["TX","CA","NY","FL","IL"]),
        "annual_income"    : int(np.clip(np.random.lognormal(10.8,0.6),25000,500000)),
        "credit_score"     : int(np.clip(np.random.normal(690,80),300,850)),
        "account_balance"  : round(random.uniform(100,150000),2),
        "product_type"     : random.choice(["CHECKING","SAVINGS","CHECKING_SAVINGS","PREMIUM"]),
        "customer_segment" : random.choice(["RETAIL","AFFLUENT","MASS_MARKET"]),
        "onboard_date"     : random_date(datetime(2015,1,1), END).strftime("%Y-%m-%d"),
        "is_active"        : random.choices([1,0],[0.92,0.08])[0],
        "branch_id"        : f"BR{random.randint(1,25):03d}",
        "source_system"    : "CORE_BANKING_v4.2",
        "created_at"       : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    })

df_customers = pd.DataFrame(customers)
df_customers.to_csv("customers.csv", index=False)
print(f"✅ customers.csv — {len(df_customers):,} rows | {len(df_customers.columns)} columns")
print(df_customers.head(3))

✅ customers.csv — 5,000 rows | 16 columns
  customer_id first_name last_name  age                      email state  \
0  CUST000001   Danielle   Johnson   48         john21@example.net    TX   
1  CUST000002        Joy   Gardner   63       fjohnson@example.org    IL   
2  CUST000003      Jesse    Guzman   64  jennifermiles@example.com    IL   

   annual_income  credit_score  account_balance      product_type  \
0          45118           741          3849.11  CHECKING_SAVINGS   
1          42595           671         13132.13           PREMIUM   
2          77688           652         90342.61           SAVINGS   

  customer_segment onboard_date  is_active branch_id      source_system  \
0           RETAIL   2018-10-18          1     BR004  CORE_BANKING_v4.2   
1           RETAIL   2015-07-05          1     BR008  CORE_BANKING_v4.2   
2      MASS_MARKET   2022-02-19          1     BR019  CORE_BANKING_v4.2   

            created_at  
0  2026-05-28 18:59:13  
1  2026-05-28 18:59:13  


In [7]:
cust_ids = [c["customer_id"] for c in customers]

loans = []
for i, cid in enumerate(random.sample(cust_ids, 3200), 1):
    loan_type = random.choices(
        ["MORTGAGE","AUTO","PERSONAL","CREDIT_LINE"],
        weights=[0.35,0.25,0.25,0.15]
    )[0]
    amount_map = {
        "MORTGAGE"   : (80000, 600000),
        "AUTO"       : (8000,  65000),
        "PERSONAL"   : (1000,  50000),
        "CREDIT_LINE": (500,   25000)
    }
    lo, hi = amount_map[loan_type]
    amt  = round(random.uniform(lo, hi), 2)
    orig = random_date(datetime(2018,1,1), datetime(2023,6,1))
    term = random.choice([12,24,36,48,60,84,120,180,360])

    loans.append({
        "loan_id"            : f"LN{i:07d}",
        "customer_id"        : cid,
        "loan_type"          : loan_type,
        "loan_amount"        : amt,
        "outstanding_balance": round(amt * random.uniform(0.1,1.0), 2),
        "interest_rate"      : round(random.uniform(3.5,24.99), 2),
        "term_months"        : term,
        "delinquency_status" : random.choices(
            ["CURRENT","30_DPD","60_DPD","90_DPD","CLOSED"],
            weights=[0.72,0.10,0.06,0.04,0.08]
        )[0],
        "origination_date"   : orig.strftime("%Y-%m-%d"),
        "monthly_payment"    : round(amt/term*random.uniform(1.0,1.3), 2),
        "branch_id"          : f"BR{random.randint(1,25):03d}",
        "source_system"      : "LOAN_ORIGINATION_v2.1",
        "created_at"         : orig.strftime("%Y-%m-%d %H:%M:%S"),
    })

df_loans = pd.DataFrame(loans)
df_loans.to_csv("loans.csv", index=False)
print(f"✅ loans.csv — {len(df_loans):,} rows | {len(df_loans.columns)} columns")
print(df_loans.head(3))

✅ loans.csv — 3,200 rows | 13 columns
     loan_id customer_id    loan_type  loan_amount  outstanding_balance  \
0  LN0000001  CUST004414  CREDIT_LINE      9080.27              5254.82   
1  LN0000002  CUST001041     PERSONAL     17586.99             14312.77   
2  LN0000003  CUST000991     PERSONAL     45096.07             36759.18   

   interest_rate  term_months delinquency_status origination_date  \
0          16.46           36            CURRENT       2019-02-27   
1          23.53           36             CLOSED       2021-01-12   
2           7.42          180            CURRENT       2022-03-04   

   monthly_payment branch_id          source_system           created_at  
0           310.87     BR010  LOAN_ORIGINATION_v2.1  2019-02-27 02:15:10  
1           627.37     BR015  LOAN_ORIGINATION_v2.1  2021-01-12 15:06:02  
2           274.46     BR009  LOAN_ORIGINATION_v2.1  2022-03-04 12:29:56  


In [8]:
txns = []
for i in range(1, 50001):
    cid  = random.choice(cust_ids)
    date = random_date(START, END)
    is_anomaly = random.random() < 0.03
    amt  = round(random.uniform(8000,85000),2) if is_anomaly else round(random.uniform(1,4500),2)
    hour = random.choice([1,2,3]) if is_anomaly else random.choices(
        range(24), weights=[1,1,1,1,1,1,2,4,6,7,8,8,7,7,8,8,7,7,6,5,4,3,2,1]
    )[0]
    dt = date.replace(hour=hour, minute=random.randint(0,59))

    txns.append({
        "transaction_id"       : f"TXN{i:09d}",
        "customer_id"          : cid,
        "transaction_date"     : dt.strftime("%Y-%m-%d"),
        "transaction_datetime" : dt.strftime("%Y-%m-%d %H:%M:%S"),
        "transaction_type"     : random.choices(
            ["DEBIT","CREDIT","TRANSFER","ATM_WITHDRAWAL","ONLINE_PAYMENT"],
            weights=[0.30,0.25,0.15,0.10,0.20]
        )[0],
        "transaction_amount"   : amt,
        "transaction_category" : random.choice([
            "GROCERIES","UTILITIES","RENT","DINING",
            "TRAVEL","HEALTHCARE","RETAIL","SALARY_CREDIT","ATM"
        ]),
        "channel"              : random.choices(
            ["MOBILE_APP","ONLINE_BANKING","ATM","BRANCH","POS_TERMINAL"],
            weights=[0.38,0.28,0.14,0.08,0.12]
        )[0],
        "is_flagged"           : 1 if is_anomaly else 0,
        "currency"             : "USD",
        "source_system"        : "TRANSACTION_LEDGER_v3.5",
        "created_at"           : dt.strftime("%Y-%m-%d %H:%M:%S"),
    })

df_transactions = pd.DataFrame(txns)
df_transactions.to_csv("transactions.csv", index=False)

flagged = df_transactions['is_flagged'].sum()
print(f"✅ transactions.csv — {len(df_transactions):,} rows | {len(df_transactions.columns)} columns")
print(f"   Anomalous transactions : {flagged:,} ({flagged/len(df_transactions)*100:.1f}%)")
print(df_transactions.head(3))

✅ transactions.csv — 50,000 rows | 12 columns
   Anomalous transactions : 1,476 (3.0%)
  transaction_id customer_id transaction_date transaction_datetime  \
0   TXN000000001  CUST004903       2023-12-21  2023-12-21 20:37:36   
1   TXN000000002  CUST000388       2021-12-09  2021-12-09 16:48:41   
2   TXN000000003  CUST003585       2022-01-25  2022-01-25 00:45:51   

  transaction_type  transaction_amount transaction_category         channel  \
0         TRANSFER             2859.32               DINING  ONLINE_BANKING   
1            DEBIT             4347.73        SALARY_CREDIT      MOBILE_APP   
2         TRANSFER             1339.20            GROCERIES      MOBILE_APP   

   is_flagged currency            source_system           created_at  
0           0      USD  TRANSACTION_LEDGER_v3.5  2023-12-21 20:37:36  
1           0      USD  TRANSACTION_LEDGER_v3.5  2021-12-09 16:48:41  
2           0      USD  TRANSACTION_LEDGER_v3.5  2022-01-25 00:45:51  


In [9]:
import os
print(os.getcwd())

C:\Users\jhanv
